一、先创建脚本

In [ ]:
vim treasure.sh

按 i 进入编辑模式

In [ ]:
#!/bin/bash
clear

LOG_FILE="$HOME/treasure_hunt.log"
TOTAL=3

T1="/tmp/secret_room/.gold.txt"
T2="/var/tmp/.silver"
T3="$HOME/.hidden_cave/.diamond"

REC1="$HOME/.rec_t1"
REC2="$HOME/.rec_t2"
REC3="$HOME/.rec_t3"

log(){
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] $1" >> "$LOG_FILE"
}

clean_all(){
    rm -rf /tmp/secret_room /var/tmp/.silver "$HOME/.hidden_cave"
    rm -f "$REC1" "$REC2" "$REC3"
}

init_treasure(){
    clean_all
    mkdir -p /tmp/secret_room
    echo "Treasure 1: Gold" > "$T1"
    chmod 644 "$T1"

    echo "Treasure 2: Silver" > "$T2"
    chmod 444 "$T2"

    mkdir -p $HOME/.hidden_cave
    echo "Treasure 3: Diamond" > "$T3"
    chmod 700 "$HOME/.hidden_cave"
    chmod 600 "$T3"
}

show_status(){
    echo "========================================"
    echo "Score: $SCORE / 100"
    echo "Progress: $PROGRESS / $TOTAL"
    echo "========================================"
}

show_menu(){
    echo "========== LINUX TREASURE HUNT =========="
    echo "1. Show Rules & Hints"
    echo "2. Enter Hunt Command Line"
    echo "3. Scan Progress"
    echo "4. View Log"
    echo "5. Reset Game"
    echo "6. Exit"
    echo -n "Enter choice: "
}

show_rule(){
    clear
    echo "========== GAME RULES =========="
    echo "1. Use 'cat' to collect treasure"
    echo "2. Scan only shows what you opened"
    echo ""
    echo "========== HINTS =========="
    echo "1. Somewhere in /tmp"
    echo "2. Somewhere in /var/tmp"
    echo "3. Somewhere in your home directory"
    echo ""
    echo "Commands: ls -a, find, cat"
    read -p "Press enter to back..."
}

hunt_shell(){
    clear
    echo "===== HUNT MODE ====="
    echo "Type 'exit' to return"
    bash --rcfile <(echo '
alias cat="__cat_record"
__cat_record(){
    for f in "$@"; do
        /bin/cat "$f" 2>/dev/null
        case "$f" in
            /tmp/secret_room/.gold.txt) touch '"$REC1"' ;;
            /var/tmp/.silver)           touch '"$REC2"' ;;
            '$HOME'/.hidden_cave/.diamond) touch '"$REC3"' ;;
        esac
    done
}
')
    clear
}

scan_progress(){
    SCORE=0
    PROGRESS=0

    [ -f "$REC1" ] && { SCORE=$((SCORE+20)); PROGRESS=$((PROGRESS+1)); }
    [ -f "$REC2" ] && { SCORE=$((SCORE+30)); PROGRESS=$((PROGRESS+1)); }
    [ -f "$REC3" ] && { SCORE=$((SCORE+50)); PROGRESS=$((PROGRESS+1)); }

    clear
    echo "Scanning progress..."
    echo ""
    show_status

    if [ $PROGRESS -eq $TOTAL ]; then
        echo "Congratulations! All treasures collected!"
        log "All treasures collected"
    fi

    read -p "Press enter to continue..."
}

reset_game(){
    init_treasure
    SCORE=0
    PROGRESS=0
    log "Game reset"
    echo "Reset completed."
    sleep 1
}

reset_game

while true; do
    clear
    show_status
    show_menu
    read opt
    case $opt in
        1) show_rule ;;
        2) hunt_shell ;;
        3) scan_progress ;;
        4) clear; cat "$LOG_FILE"; read -p "Press enter..." ;;
        5) reset_game ;;
        6) clear; echo "Goodbye!"; exit 0 ;;
        *) echo "Invalid Input"; sleep 1 ;;
    esac
done

二、加执行权限

In [ ]:
chmod +x treasure.sh

三、启动游戏